# 4.75. Split Review

Reviews all clusters created by the Cluster Splitter (4.7).
Each split produces two child clusters — shown here side by side so you can
verify or correct both labels before re-running the dataset pipeline.

**Steps:**
1. Run all cells.
2. Navigate pairs with **◀ Prev / Next ▶**.
3. Correct the Part A / Part B label dropdowns if needed.
4. Click **✔ Confirm pair** to save and advance.

In [ ]:
import sys, io
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

from config import CLUSTERS_DIR
from utils.labels import Labels

In [ ]:
INV_PATH = CLUSTERS_DIR / 'inventory.csv'
inv = pd.read_csv(INV_PATH)

# All clusters produced by the splitter
split_mask = inv['label_source'].fillna('') == 'manual_split'
split_clusters = inv[split_mask].sort_values(['tilecode', 'cluster_idx']).copy()

# Pair consecutive cluster_idx values within the same tilecode
pairs = []   # list of (inv_idx_a, inv_idx_b | None)
used  = set()
for idx, row in split_clusters.iterrows():
    if idx in used:
        continue
    sibling = split_clusters[
        (split_clusters['tilecode'] == row['tilecode']) &
        (split_clusters['cluster_idx'] == row['cluster_idx'] + 1)
    ]
    if len(sibling) == 1:
        pairs.append((idx, sibling.index[0]))
        used.update([idx, sibling.index[0]])
    else:
        pairs.append((idx, None))   # orphaned child
        used.add(idx)

n_paired   = sum(1 for a, b in pairs if b is not None)
n_orphaned = sum(1 for a, b in pairs if b is None)
print(f'Split clusters : {len(split_clusters)}')
print(f'Pairs          : {n_paired}')
if n_orphaned:
    print(f'Unpaired       : {n_orphaned}  (shown individually at end)')

In [ ]:
LABEL_NAMES = {
    0: 'Unknown', 1: 'Road', 9: 'Ground', 10: 'Building', 11: 'Facade',
    15: 'Bus/tram shelter',
    30: 'Tree', 39: 'Green (other)', 40: 'Car', 44: 'Bicycle', 50: 'Person',
    60: 'Street Light', 61: 'Traffic Light', 62: 'Traffic Sign',
    65: 'Bollard', 67: 'Stop Pole', 80: 'City Bench',
    81: 'Rubbish Bin', 83: 'Large Container', 85: 'Parking Meter',
    88: 'Bicycle Rack', 99: 'Noise / False Positive',
}
LABEL_OPTIONS = [(f"{v}  ({k})", k) for k, v in sorted(LABEL_NAMES.items(), key=lambda x: x[1])]


def _sa(ax):
    ax.set_facecolor('#1a1a1a')
    for sp in ax.spines.values():
        sp.set_edgecolor('#444')
    ax.tick_params(labelsize=6, colors='grey')


def _render_cluster_axes(ax_top, ax_side, row, title_color='#aaa', label_override=None):
    """Draw top and side views of one cluster onto pre-created axes."""
    _sa(ax_top); _sa(ax_side)
    try:
        npz  = np.load(row['npz_path'], allow_pickle=False)
        xyz  = npz['xyz_centered']
        rgb  = np.clip(npz['rgb_norm'], 0, 1)
        hag  = npz['height_ag']
    except Exception as e:
        ax_top.text(0.5, 0.5, f'Cannot load:\n{e}', color='#cc4444',
                    ha='center', va='center', transform=ax_top.transAxes, fontsize=7)
        return

    pt_sz = max(2, min(12, 3000 // max(len(xyz), 1)))
    lbl_code = label_override if label_override is not None else int(row['final_label'])
    lbl_name = LABEL_NAMES.get(lbl_code, str(lbl_code))
    tile     = str(row['tilecode'])
    n_pts    = int(row['n_raw_pts'])

    ax_top.scatter(xyz[:, 0], xyz[:, 1], c=rgb, s=pt_sz, linewidths=0)
    ax_top.set_aspect('equal')
    ax_top.set_title(f'{lbl_name}  ·  {tile}  ·  {n_pts:,} pts',
                     color=title_color, fontsize=8)
    ax_top.set_xlabel('ΔX (m)', color='grey', fontsize=6)
    ax_top.set_ylabel('ΔY (m)', color='grey', fontsize=6)

    ax_side.scatter(xyz[:, 0], hag, c=rgb, s=pt_sz, linewidths=0)
    ax_side.set_xlabel('ΔX (m)', color='grey', fontsize=6)
    ax_side.set_ylabel('h ag (m)', color='grey', fontsize=6)
    ax_side.set_title('side view', color='#666', fontsize=7)


def render_pair_png(idx_a, idx_b):
    """Render side-by-side views for a split pair. Returns PNG bytes."""
    row_a = inv.loc[idx_a]

    if idx_b is not None:
        row_b = inv.loc[idx_b]
        fig = Figure(figsize=(13, 5), facecolor='#1a1a1a')
        FigureCanvasAgg(fig)
        ax_at = fig.add_subplot(2, 2, 1)
        ax_as = fig.add_subplot(2, 2, 3)
        ax_bt = fig.add_subplot(2, 2, 2)
        ax_bs = fig.add_subplot(2, 2, 4)
        _render_cluster_axes(ax_at, ax_as, row_a, title_color='#88aaff')
        _render_cluster_axes(ax_bt, ax_bs, row_b, title_color='#ffaa66')
        fig.text(0.27, 0.97, 'Part A', color='#88aaff', fontsize=10,
                 ha='center', va='top', fontweight='bold')
        fig.text(0.75, 0.97, 'Part B', color='#ffaa66', fontsize=10,
                 ha='center', va='top', fontweight='bold')
    else:
        fig = Figure(figsize=(7, 5), facecolor='#1a1a1a')
        FigureCanvasAgg(fig)
        ax_at = fig.add_subplot(2, 1, 1)
        ax_as = fig.add_subplot(2, 1, 2)
        _render_cluster_axes(ax_at, ax_as, row_a, title_color='#88aaff')
        fig.text(0.5, 0.97, 'Part A (no sibling found)', color='#ffaa44',
                 fontsize=9, ha='center', va='top')

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, facecolor='#1a1a1a', bbox_inches='tight')
    buf.seek(0)
    return buf.read()

In [ ]:
if not pairs:
    display(widgets.HTML('<span style="color:#aaffaa;font-size:14px">No split clusters found in inventory.</span>'))
else:
    # ── state ──────────────────────────────────────────────────────────────────
    state = {'pair_idx': 0}

    # ── widgets ────────────────────────────────────────────────────────────────
    nav_html   = widgets.HTML()
    btn_prev   = widgets.Button(description='◀ Prev',  layout=widgets.Layout(width='90px'))
    btn_next   = widgets.Button(description='Next ▶',  layout=widgets.Layout(width='90px'))

    preview_img = widgets.Image(format='png', layout=widgets.Layout(width='100%', max_width='860px'))

    label_a_dd = widgets.Dropdown(
        options=LABEL_OPTIONS,
        description='Part A:',
        layout=widgets.Layout(width='340px'),
        style={'description_width': '60px'},
    )
    label_b_dd = widgets.Dropdown(
        options=LABEL_OPTIONS,
        description='Part B:',
        layout=widgets.Layout(width='340px'),
        style={'description_width': '60px'},
    )
    btn_confirm = widgets.Button(
        description='✔ Confirm pair',
        button_style='success',
        layout=widgets.Layout(width='160px'),
    )
    status_html = widgets.HTML()

    label_b_box = widgets.HBox([label_b_dd])

    # ── helpers ────────────────────────────────────────────────────────────────
    def _load_pair(pair_idx):
        idx_a, idx_b = pairs[pair_idx]
        row_a = inv.loc[idx_a]
        lbl_a = int(row_a['final_label']) if pd.notna(row_a['final_label']) else 0
        label_a_dd.value = lbl_a if lbl_a in dict(LABEL_OPTIONS).values() else 0

        if idx_b is not None:
            row_b = inv.loc[idx_b]
            lbl_b = int(row_b['final_label']) if pd.notna(row_b['final_label']) else 0
            label_b_dd.value = lbl_b if lbl_b in dict(LABEL_OPTIONS).values() else 0
            label_b_box.layout.display = ''
        else:
            label_b_box.layout.display = 'none'

        preview_img.value = render_pair_png(idx_a, idx_b)
        nav_html.value = (
            f'<span style="color:#aaa;font-size:12px">'
            f'Pair <b>{pair_idx + 1}</b> / {len(pairs)}')
        status_html.value = ''

    def _save_current():
        pair_idx = state['pair_idx']
        idx_a, idx_b = pairs[pair_idx]
        inv.at[idx_a, 'final_label']        = int(label_a_dd.value)
        inv.at[idx_a, 'label_source_final'] = 'manual_split_reviewed'
        if idx_b is not None:
            inv.at[idx_b, 'final_label']        = int(label_b_dd.value)
            inv.at[idx_b, 'label_source_final'] = 'manual_split_reviewed'
        inv.to_csv(INV_PATH, index=False)

    # ── event handlers ─────────────────────────────────────────────────────────
    def _on_confirm(_):
        _save_current()
        status_html.value = '<span style="color:#aaffaa">✔ Saved</span>'
        pi = state['pair_idx']
        if pi + 1 < len(pairs):
            state['pair_idx'] = pi + 1
            _load_pair(state['pair_idx'])
        else:
            status_html.value = '<span style="color:#aaffaa">✔ All pairs reviewed.</span>'

    def _on_prev(_):
        if state['pair_idx'] > 0:
            state['pair_idx'] -= 1
            _load_pair(state['pair_idx'])

    def _on_next(_):
        if state['pair_idx'] + 1 < len(pairs):
            state['pair_idx'] += 1
            _load_pair(state['pair_idx'])

    btn_confirm.on_click(_on_confirm)
    btn_prev.on_click(_on_prev)
    btn_next.on_click(_on_next)

    # ── layout ─────────────────────────────────────────────────────────────────
    display(widgets.VBox([
        widgets.HBox([btn_prev, btn_next, nav_html],
                     layout=widgets.Layout(gap='8px', align_items='center')),
        preview_img,
        widgets.HBox([label_a_dd, label_b_box],
                     layout=widgets.Layout(gap='16px')),
        widgets.HBox([btn_confirm, status_html],
                     layout=widgets.Layout(gap='12px', align_items='center')),
    ]))

    _load_pair(0)